In [3]:
import pandas as pd
cols = pd.read_parquet(r"D:\AI\Real projects\Academic_Advisor\data\model_data\df_train_final.parquet").columns
print("diploma_type_bucket" in cols, "course_pass_rate_historical" in cols)

True True


In [ ]:
import json

import lightgbm as lgb
import pandas as pd

import src.model_training as mt
from src.paths import MODEL_DATA_DIR, MODELS_DIR

RUN_DIR = MODELS_DIR / "runs" / "2026-07-16_1025__new-difficulty-logic"
DATA_VERSION_DIR = (
    MODEL_DATA_DIR
    / "versions"
    / "2026-07-16_095738__b2_temporal_course_stats"
)

model = lgb.Booster(model_file=str(RUN_DIR / "m1_pass_model.lgbm"))
with (RUN_DIR / "feature_contract.json").open(encoding="utf-8") as f:
    contract = json.load(f)

categorical_levels = contract["categorical_levels"]
df_valid = pd.read_parquet(DATA_VERSION_DIR / "df_valid_final.parquet")
df_test = pd.read_parquet(DATA_VERSION_DIR / "df_test_final.parquet")

mt._THRESHOLDS = [0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
mt.print_threshold_table(model, df_valid, df_test, categorical_levels)
mt.stratified_threshold_table(model, df_test, categorical_levels)


=== M1 THRESHOLD TABLE (fail-class P/R/F1) ===
  Thresholds: [0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
  Decision on VALID only — TEST shown for consistency comparison.

  [VALID]  thr    fail_P    fail_R   fail_F1
  ----------------------------------------------
  [VALID]  0.7   0.4457    0.1473   0.2214
  [VALID]  0.7   0.4097    0.2212   0.2873
  [VALID]  0.8   0.3629    0.3196   0.3399
  [VALID]  0.8   0.3219    0.4396   0.3717
  [VALID]  0.8   0.2764    0.5848   0.3754
  [VALID]  0.9   0.2366    0.7320   0.3577

  [TEST ]  thr    fail_P    fail_R   fail_F1
  ----------------------------------------------
  [TEST ]  0.7   0.3671    0.1199   0.1808
  [TEST ]  0.7   0.3298    0.1783   0.2315
  [TEST ]  0.8   0.2960    0.2616   0.2777
  [TEST ]  0.8   0.2694    0.3754   0.3137
  [TEST ]  0.8   0.2334    0.5242   0.3230
  [TEST ]  0.9   0.2000    0.6912   0.3102

  (No threshold auto-selected — choose after reviewing the table above.)
=== END THRESHOLD TABLE ===


=== M1 SEGMENTED THRESHOLD T

In [2]:
from sklearn.metrics import roc_auc_score, brier_score_loss

X_test, y_test = mt.prepare_X_y(df_test, categorical_levels)

proba = model.predict(X_test)

print("AUC:", roc_auc_score(y_test, proba))
print("Brier:", brier_score_loss(y_test, proba))

NameError: name 'mt' is not defined